In [7]:
import lancedb

db = lancedb.connect("article_db")

In [8]:
data = []
for i in range(1,7):
    with open(f"data/article{i}.md", "r", encoding="utf8") as f:
        d = f.read()
        data.append(d)

len(data)

6

In [9]:
import pandas as pd

df = pd.DataFrame(data=data, columns=["article"])
df

,article
0,Gothenburg (/ˈɡɒθənbɜːrɡ/ ⓘ GOTH-ən-burg; Swed...
1,"The cat (Felis catus), also called domestic ca..."
2,The dog (Canis familiaris or Canis lupus famil...
3,"Rabbits, or bunnies, are small mammals in the ..."
4,Stockholm (/ˈstɒkhoʊ(l)m/;[10] Swedish: [ˈstɔ̂...
5,"Monty Python, also known as the Pythons,[2][3]..."


In [22]:
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry


model = get_registry().get("gemini-text").create(name="gemini-embedding-001")


class ArticleSchema(LanceModel):
    article: str = model.SourceField()
    vector: Vector(3072) = model.VectorField()
    
article_table = db.create_table("articles", schema=ArticleSchema, exist_ok=True)
article_table

LanceTable(name='articles', version=2, _conn=LanceDBConnection(uri='c:\\Users\\organ\\Repos\\ml_ai\\ai_engineering_pontus_agren_grundstrom\\exercises\\6_chatbot\\article_db'))

In [11]:
article_table.add(df)
article_table.to_pandas()

c:\Users\organ\Repos\ml_ai\ai_engineering_pontus_agren_grundstrom\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\organ\AppData\Local\Programs\Python\Python312\Lib\importlib\__init__.py:90: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  return _bootstrap._gcd_import(name[level:], package, level)


,article,vector
0,Gothenburg (/ˈɡɒθənbɜːrɡ/ ⓘ GOTH-ən-burg; Swed...,"[-0.010164232, 0.0067330464, -0.0068469923, -0..."
1,"The cat (Felis catus), also called domestic ca...","[-0.014599158, 0.017405435, 0.012461907, -0.05..."
2,The dog (Canis familiaris or Canis lupus famil...,"[-0.010156812, 0.024125773, 0.006588903, -0.04..."
3,"Rabbits, or bunnies, are small mammals in the ...","[-0.017020063, 0.02435889, -0.0011976677, -0.0..."
4,Stockholm (/ˈstɒkhoʊ(l)m/;[10] Swedish: [ˈstɔ̂...,"[-0.034094103, 0.0038274885, -0.015730219, -0...."
5,"Monty Python, also known as the Pythons,[2][3]...","[-0.025864772, -0.0028935557, 0.010479224, -0...."


In [19]:
article_table.search("give me an article about predators").limit(3).to_pandas()

,article,vector,_distance
0,"The cat (Felis catus), also called domestic ca...","[-0.014599158, 0.017405435, 0.012461907, -0.05...",0.685947
1,The dog (Canis familiaris or Canis lupus famil...,"[-0.010156812, 0.024125773, 0.006588903, -0.04...",0.766412
2,"Rabbits, or bunnies, are small mammals in the ...","[-0.017020063, 0.02435889, -0.0011976677, -0.0...",0.767143


In [20]:
from lancedb import rerankers

reranker = rerankers.RRFReranker()

query = "give me an article about predators"

results = article_table.search(
    query,
    query_type="hybrid",
    vector_column_name="vector",
    fts_columns="article",
).rerank(reranker).limit(8).to_pandas()

# somewhat different to before
results

,article,vector,_relevance_score
0,"The cat (Felis catus), also called domestic ca...","[-0.014599158, 0.017405435, 0.012461907, -0.05...",0.032522
1,"Rabbits, or bunnies, are small mammals in the ...","[-0.017020063, 0.02435889, -0.0011976677, -0.0...",0.031746
2,Gothenburg (/ˈɡɒθənbɜːrɡ/ ⓘ GOTH-ən-burg; Swed...,"[-0.010164232, 0.0067330464, -0.0068469923, -0...",0.031545
3,"Monty Python, also known as the Pythons,[2][3]...","[-0.025864772, -0.0028935557, 0.010479224, -0....",0.031250
4,The dog (Canis familiaris or Canis lupus famil...,"[-0.010156812, 0.024125773, 0.006588903, -0.04...",0.016129
5,Stockholm (/ˈstɒkhoʊ(l)m/;[10] Swedish: [ˈstɔ̂...,"[-0.034094103, 0.0038274885, -0.015730219, -0....",0.015385
